In [11]:
from sklearn.linear_model import LogisticRegression
import pandas as pd

# Select features and the true categorical controls from your dataset
features = ['missing_repair_cost']
control_cols = ['vehicleType', 'damageDescription']  # Updated to match your exact columns!

# Drop rows missing our categorical controls for a clean regression run
df_reg = df.dropna(subset=control_cols).copy()

# One-hot encode our categorical variables
X = pd.get_dummies(df_reg[features + control_cols], columns=control_cols, drop_first=True)
y = df_reg['is_pure_sale']

# Fit the model
lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

# Match coefficients back to feature names
coefficients = dict(zip(X.columns, lr.coef_[0]))

print("--- REGRESSION RIGOR CHECK ---")
print(f"Coefficient for 'missing_repair_cost': {coefficients['missing_repair_cost']:.4f}")

--- REGRESSION RIGOR CHECK ---
Coefficient for 'missing_repair_cost': -1.6724


In [5]:
# 1. Create a binary flag for missing repair costs
df['missing_repair_cost'] = (df['repairCost'] == 0).astype(int)

# 2. Calculate clearance rates grouped by this flag
anomaly_report = df.groupby('missing_repair_cost').agg(
    lot_count=('is_pure_sale', 'count'),
    clearance_rate=('is_pure_sale', 'mean')
).reset_index()

# Convert clearance to percentage for readability
anomaly_report['clearance_rate'] *= 100

print("--- THE ANOMALY REPORT ---")
print(anomaly_report.to_string(index=False))

--- THE ANOMALY REPORT ---
 missing_repair_cost  lot_count  clearance_rate
                   0      18922       83.077899
                   1       5608       35.342368


In [7]:
# Look for the exact column names in your dataset
print("Title columns:", [col for col in df.columns if 'title' in col.lower()])
print("Damage columns:", [col for col in df.columns if 'damage' in col.lower()])
print("Vehicle columns:", [col for col in df.columns if 'veh' in col.lower()])

Title columns: ['saleTitleState', 'saleTitleType']
Damage columns: ['damageDescription', 'secondaryDamage']
Vehicle columns: ['vehicleType']


In [12]:
# Group by the correct title column found in your diagnostic cell
correct_title_col = 'saleTitleType' 

title_analysis = df.groupby(correct_title_col).agg(
    total_lots=('is_pure_sale', 'count'),
    pct_missing_repair_cost=('missing_repair_cost', 'mean'),
    clearance_rate=('is_pure_sale', 'mean')
).reset_index()

# Format as percentages
title_analysis['pct_missing_repair_cost'] *= 100
title_analysis['clearance_rate'] *= 100

print("--- TITLE TYPE DEEP DIVE ---")
print(title_analysis.sort_values(by='total_lots', ascending=False).head(10).to_string(index=False))

--- TITLE TYPE DEEP DIVE ---
saleTitleType  total_lots  pct_missing_repair_cost  clearance_rate
           SC        6313                17.075875       86.028829
           ST        4839                13.267204       84.149618
           CT        3785                62.007926       26.420079
           RB        2075                 4.674699       85.542169
           SV        1538                 9.297789       81.729519
           S1         861                10.452962       90.011614
           BP         630                 6.666667       88.571429
           CD         612                 8.660131       74.509804
           BS         612                21.732026       67.647059
           SM         411                 9.975669       58.394161
